In [1]:
import numpy as np
import joblib
import pandas as pd

from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [3]:
X_train = np.load("../dataset/X_train.npy")
X_test = np.load("../dataset/X_test.npy")

y_train = np.load("../dataset/y_train.npy")
y_test = np.load("../dataset/y_test.npy")

print("✓ Data loaded successfully!")
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

✓ Data loaded successfully!
Training set shape: (44756, 187)
Testing set shape: (11189, 187)


In [4]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)

ada = AdaBoostClassifier(random_state=42)

knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

print("✓ Base models initialized!")

✓ Base models initialized!


In [5]:
hard_voting = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('gb', gb),
        ('knn', knn)
    ],
    voting='hard'
)

print("Training Hard Voting Ensemble...")
hard_voting.fit(X_train, y_train)

y_pred_hard = hard_voting.predict(X_test)
print("✓ Hard Voting trained and predictions made!")

Training Hard Voting Ensemble...
✓ Hard Voting trained and predictions made!


In [6]:
hard_results = {
    "Model": "Hard Voting",
    "Accuracy": accuracy_score(y_test, y_pred_hard),
    "Precision": precision_score(y_test, y_pred_hard),
    "Recall": recall_score(y_test, y_pred_hard),
    "F1": f1_score(y_test, y_pred_hard)
}

print("Hard Voting Results:")
for k, v in hard_results.items():
    if k != "Model":
        print(f"  {k}: {v:.4f}")

Hard Voting Results:
  Accuracy: 0.9612
  Precision: 0.9600
  Recall: 0.9393
  F1: 0.9495


In [7]:
soft_voting = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('gb', gb),
        ('ada', ada)
    ],
    voting='soft'
)

print("Training Soft Voting Ensemble...")
soft_voting.fit(X_train, y_train)

y_pred_soft = soft_voting.predict(X_test)
print("✓ Soft Voting trained and predictions made!")

Training Soft Voting Ensemble...
✓ Soft Voting trained and predictions made!


In [8]:
soft_results = {
    "Model": "Soft Voting",
    "Accuracy": accuracy_score(y_test, y_pred_soft),
    "Precision": precision_score(y_test, y_pred_soft),
    "Recall": recall_score(y_test, y_pred_soft),
    "F1": f1_score(y_test, y_pred_soft)
}

print("Soft Voting Results:")
for k, v in soft_results.items():
    if k != "Model":
        print(f"  {k}: {v:.4f}")

Soft Voting Results:
  Accuracy: 0.9636
  Precision: 0.9616
  Recall: 0.9441
  F1: 0.9528


In [9]:
stacking = StackingClassifier(
    estimators=[
        ('rf', rf),
        ('gb', gb),
        ('ada', ada)
    ],
    final_estimator=LogisticRegression(random_state=42),
    cv=5,
    n_jobs=-1
)

print("Training Stacking Ensemble (this may take a minute)...")
stacking.fit(X_train, y_train)

y_pred_stack = stacking.predict(X_test)
print("✓ Stacking Ensemble trained and predictions made!")

Training Stacking Ensemble (this may take a minute)...
✓ Stacking Ensemble trained and predictions made!


In [10]:
stack_results = {
    "Model": "Stacking",
    "Accuracy": accuracy_score(y_test, y_pred_stack),
    "Precision": precision_score(y_test, y_pred_stack),
    "Recall": recall_score(y_test, y_pred_stack),
    "F1": f1_score(y_test, y_pred_stack)
}

print("Stacking Results:")
for k, v in stack_results.items():
    if k != "Model":
        print(f"  {k}: {v:.4f}")

Stacking Results:
  Accuracy: 0.9658
  Precision: 0.9620
  Recall: 0.9494
  F1: 0.9557


In [11]:
joblib.dump(hard_voting, "../models/hard_voting.pkl")
joblib.dump(soft_voting, "../models/soft_voting.pkl")
joblib.dump(stacking, "../models/stacking.pkl")

print("✓ All ensemble models saved to the 'models' folder!")

✓ All ensemble models saved to the 'models' folder!


In [13]:
# 1. Import the missing confusion_matrix along with roc_auc_score
from sklearn.metrics import roc_auc_score, confusion_matrix

# 2. Define the advanced evaluation function
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    # Basic metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Confusion matrix metrics
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp)
    fpr = fp / (fp + tn)
    
    # ROC-AUC
    auc = "N/A"
    if hasattr(model, "predict_proba"):
        try:
            y_prob = model.predict_proba(X_test)[:, 1]
            auc = round(roc_auc_score(y_test, y_prob), 4)
        except Exception:
            auc = "N/A" # Hard Voting doesn't support probabilities
            
    return {
        "Model": name,
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1": round(f1, 4),
        "Specificity": round(specificity, 4),
        "FPR": round(fpr, 4),
        "AUC": auc
    }

# 3. Load all models (Baselines + Ensembles)
print("Loading models for final evaluation...")
models_to_evaluate = {
    "Random Forest": joblib.load("../models/Random_Forest.pkl"),
    "AdaBoost": joblib.load("../models/AdaBoost.pkl"),
    "Gradient Boosting": joblib.load("../models/Gradient_Boosting.pkl"),
    "KNN": joblib.load("../models/KNN.pkl"),
    "Naive Bayes": joblib.load("../models/Naive_Bayes.pkl"),
    "Hard Voting": joblib.load("../models/hard_voting.pkl"),
    "Soft Voting": joblib.load("../models/soft_voting.pkl"),
    "Stacking": joblib.load("../models/stacking.pkl")
}

# 4. Evaluate all models
final_results = []
for name, model in models_to_evaluate.items():
    print(f"Evaluating {name}...")
    metrics = evaluate_model(name, model, X_test, y_test)
    final_results.append(metrics)

# 5. Create the Final Thesis Table
thesis_table = pd.DataFrame(final_results)

# Sort by F1-Score to highlight the best model
thesis_table = thesis_table.sort_values("F1", ascending=False).reset_index(drop=True)

# Display the table
print("\n" + "="*50)
print("FINAL THESIS COMPARISON TABLE")
print("="*50)
display(thesis_table)

Loading models for final evaluation...
Evaluating Random Forest...
Evaluating AdaBoost...
Evaluating Gradient Boosting...
Evaluating KNN...
Evaluating Naive Bayes...
Evaluating Hard Voting...
Evaluating Soft Voting...
Evaluating Stacking...

FINAL THESIS COMPARISON TABLE


,Model,Accuracy,Precision,Recall,F1,Specificity,FPR,AUC
0,Stacking,0.9658,0.9620,0.9494,0.9557,0.9762,0.0238,0.994
1,Random Forest,0.9652,0.9665,0.9432,0.9547,0.9792,0.0208,0.9938
2,Soft Voting,0.9636,0.9616,0.9441,0.9528,0.9760,0.0240,0.9927
3,Hard Voting,0.9612,0.9600,0.9393,0.9495,0.9751,0.0249,N/A
4,Gradient Boosting,0.9501,0.9407,0.9303,0.9355,0.9627,0.0373,0.9885
5,KNN,0.9153,0.9267,0.8491,0.8862,0.9573,0.0427,0.965
6,AdaBoost,0.8936,0.8741,0.8482,0.8610,0.9224,0.0776,0.9654
7,Naive Bayes,0.6804,0.9987,0.1778,0.3018,0.9999,0.0001,0.6


In [14]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
import joblib

# Re-instantiate all base models to ensure they are fresh
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
ada = AdaBoostClassifier(random_state=42)
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
nb = GaussianNB()

print("Training Stacking B (All 5 Models)...")

stacking_all = StackingClassifier(
    estimators=[
        ('rf', rf),
        ('gb', gb),
        ('ada', ada),
        ('knn', knn),
        ('nb', nb)
    ],
    final_estimator=LogisticRegression(random_state=42),
    cv=5,
    n_jobs=-1
)

stacking_all.fit(X_train, y_train)
y_pred_stack_all = stacking_all.predict(X_test)

# Save the model
joblib.dump(stacking_all, "../models/stacking_all.pkl")
print("✓ Stacking B trained and saved!")

Training Stacking B (All 5 Models)...
✓ Stacking B trained and saved!


In [15]:
import time
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# 1. Load test data (needed to measure prediction time)
X_test = np.load("../dataset/X_test.npy")
y_test = np.load("../dataset/y_test.npy")

# 2. Define the function to measure EVERYTHING
def full_evaluation(name, filepath, X_test, y_test):
    # Load the saved model
    model = joblib.load(filepath)
    
    # --- A. Measure Model Size ---
    size_mb = round(os.path.getsize(filepath) / (1024 * 1024), 2)
    
    # --- B. Measure Prediction Time ---
    start_time = time.time()
    y_pred = model.predict(X_test)
    end_time = time.time()
    
    total_time = end_time - start_time
    time_per_packet_ms = round((total_time / len(X_test)) * 1000, 4)
    
    # --- C. Calculate Performance Metrics ---
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp)
    fpr = fp / (fp + tn)
    
    auc = "N/A"
    if hasattr(model, "predict_proba"):
        try:
            y_prob = model.predict_proba(X_test)[:, 1]
            auc = round(roc_auc_score(y_test, y_prob), 4)
        except Exception:
            auc = "N/A"
            
    return {
        "Model": name,
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1": round(f1, 4),
        "Specificity": round(specificity, 4),
        "FPR": round(fpr, 4),
        "AUC": auc,
        "Size (MB)": size_mb,
        "Time/pkt (ms)": time_per_packet_ms
    }

# 3. List all your saved models
model_files = {
    "Random Forest": "../models/Random_Forest.pkl",
    "AdaBoost": "../models/AdaBoost.pkl",
    "Gradient Boosting": "../models/Gradient_Boosting.pkl",
    "KNN": "../models/KNN.pkl",
    "Naive Bayes": "../models/Naive_Bayes.pkl",
    "Hard Voting": "../models/hard_voting.pkl",
    "Soft Voting": "../models/soft_voting.pkl",
    "Stacking A (Top 3)": "../models/stacking.pkl",
    "Stacking B (All 5)": "../models/stacking_all.pkl"
}

# 4. Run the evaluation
print("⏳ Measuring Performance and Efficiency for all models...")
final_results = []
for name, path in model_files.items():
    print(f"  - Evaluating {name}...")
    final_results.append(full_evaluation(name, path, X_test, y_test))

# 5. Create the Master Thesis Table
master_table = pd.DataFrame(final_results)
master_table = master_table.sort_values("F1", ascending=False).reset_index(drop=True)

print("\n" + "="*80)
print("MASTER THESIS TABLE (Performance + Efficiency)")
print("="*80)
display(master_table)

# Save to CSV for your paper
master_table.to_csv("../results/master_thesis_table.csv", index=False)
print("\n✓ Saved to ../results/master_thesis_table.csv")

⏳ Measuring Performance and Efficiency for all models...
  - Evaluating Random Forest...
  - Evaluating AdaBoost...
  - Evaluating Gradient Boosting...
  - Evaluating KNN...
  - Evaluating Naive Bayes...
  - Evaluating Hard Voting...
  - Evaluating Soft Voting...
  - Evaluating Stacking A (Top 3)...
  - Evaluating Stacking B (All 5)...

MASTER THESIS TABLE (Performance + Efficiency)


,Model,Accuracy,Precision,Recall,F1,Specificity,FPR,AUC,Size (MB),Time/pkt (ms)
0,Stacking A (Top 3),0.9658,0.9620,0.9494,0.9557,0.9762,0.0238,0.994,40.82,0.0526
1,Stacking B (All 5),0.9658,0.9620,0.9494,0.9557,0.9762,0.0238,0.9939,105.03,0.5138
2,Random Forest,0.9652,0.9665,0.9432,0.9547,0.9792,0.0208,0.9938,40.65,0.0137
3,Soft Voting,0.9636,0.9616,0.9441,0.9528,0.9760,0.0240,0.9927,40.81,0.0917
4,Hard Voting,0.9612,0.9600,0.9393,0.9495,0.9751,0.0249,N/A,104.98,0.4162
5,Gradient Boosting,0.9501,0.9407,0.9303,0.9355,0.9627,0.0373,0.9885,0.13,0.0044
6,KNN,0.9153,0.9267,0.8491,0.8862,0.9573,0.0427,0.965,64.20,0.3861
7,AdaBoost,0.8936,0.8741,0.8482,0.8610,0.9224,0.0776,0.9654,0.03,0.0605
8,Naive Bayes,0.6804,0.9987,0.1778,0.3018,0.9999,0.0001,0.6,0.01,0.0075



✓ Saved to ../results/master_thesis_table.csv
